# AI-Based Lead Scoring & Analytics System
This notebook trains and evaluates ML models for lead conversion prediction using the Bank Marketing dataset.
**Dataset source:** UCI Machine Learning Repository — Bank Marketing Data Set (https://archive.ics.uci.edu/dataset/222/bank+marketing).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report, RocCurveDisplay
import joblib
sns.set_theme(style='whitegrid')

In [ ]:
# Load dataset
data_path = Path('../data/bank.csv')
df = pd.read_csv(data_path)
df.head()

In [ ]:
# Basic EDA
print('Shape:', df.shape)
print(df['y'].value_counts(normalize=True))
display(df.describe(include='all').transpose().head(20))

In [ ]:
# Example visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['age'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age Distribution')
sns.countplot(data=df, x='y', ax=axes[1], palette='viridis')
axes[1].set_title('Target Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Data cleaning
df = df.drop_duplicates().copy()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()
df.isna().sum()

In [ ]:
# Feature/target split
X = df.drop(columns=['y'])
y = (df['y'].str.lower() == 'yes').astype(int)
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
# Preprocessing and model pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
}

In [ ]:
# Train and evaluate models
results = []
trained_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    results.append(metrics)
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df

In [ ]:
# Plot ROC for best model
best_model_name = results_df.iloc[0]['model']
best_model = trained_models[best_model_name]
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f'ROC Curve - {best_model_name}')
plt.show()
print('Selected model:', best_model_name)

In [ ]:
# Save model bundle for backend inference
model_bundle = {
    'model': best_model,
    'feature_columns': X.columns.tolist(),
    'selected_model': best_model_name,
    'metrics': results_df.to_dict(orient='records')
}
joblib.dump(model_bundle, '../backend/model.pkl')
print('Model saved to ../backend/model.pkl')